In [ ]:
import pandas as pd

# 文件的本地路径（使用 r 前缀是为了防止路径中的反斜杠 \ 被转义）
file_path = r".\陈祖武先生捐赠书籍整理书目.xlsx"

try:
    # 只读取第一行（默认它为表头）
    # 如果你想把第一行当做普通数据而不是表头，可以加上参数 header=None
    df = pd.read_excel(file_path, nrows=0) 
    
    print("成功读取！该Excel文件的首行（列名）包含以下内容：")
    
    # 获取并打印所有的列名
    columns = df.columns.tolist()
    for i, col in enumerate(columns, 1):
        print(f"第 {i} 列: {col}")
        
except FileNotFoundError:
    print(f"错误：找不到文件，请检查路径 {file_path} 是否完全正确。")
except Exception as e:
    print(f"读取时发生错误：{e}")

成功读取！该Excel文件的首行（列名）包含以下内容：
第 1 列: 序号
第 2 列: 题名
第 3 列: 索书号
第 4 列: 册数
第 5 列: 单价
第 6 列: 出版者
第 7 列: 责任者
第 8 列: 收藏库室
第 9 列: 附件形式
（手写、信件、卡片等）
第 10 列: 附件内容
第 11 列: 类型
第 12 列: 著者简介
第 13 列: 其他


In [ ]:
import pandas as pd

# 1. 定义原文件路径和新文件的保存路径
input_file = r".\陈祖武先生捐赠书籍整理书目.xlsx"
output_file = r".\有附件的书目筛选结果.xlsx" 

try:
    print("正在读取全部数据，这可能需要几秒钟，请稍候...")
    df = pd.read_excel(input_file)
    
    # 清洗列名：去掉隐藏的换行符和首尾空格
    df.columns = df.columns.str.replace('\n', '').str.replace('\r', '').str.strip()
    
    target_columns = ['附件形式（手写、信件、卡片等）', '附件内容', '类型']
    
    # 【关键修改】筛选数据：删除这三列【全都是】空白的行
    # how='all' 的意思是，只有当这三列同时为空时，才剔除该行。
    # 换言之：但凡这三列中有一个列有值，这行就会被保留！
    filtered_df = df.dropna(subset=target_columns, how='all')
    
    # 保存结果
    filtered_df.to_excel(output_file, index=False)
    
    print("\n--- 处理完成！---")
    print(f"原始数据总计: {len(df)} 行")
    print(f"符合条件的数据: {len(filtered_df)} 行")
    print(f"新文件已成功保存至: {output_file}")

except PermissionError:
    print("\n错误：写入失败。请确保你没有在Excel里打开“有附件的书目筛选结果.xlsx”。如果打开了，请关闭后再运行！")
except KeyError as e:
    print(f"\n错误：找不到列名 {e}。")
    if 'df' in locals():
        print("当前实际读取到的列名有：", df.columns.tolist())
except Exception as e:
    print(f"\n处理时发生未知错误：{e}")

正在读取全部数据，这可能需要几秒钟，请稍候...


d:\Anaconda\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")



--- 处理完成！---
原始数据总计: 4848 行
符合条件的数据: 1264 行
新文件已成功保存至: C:\Users\ASUS\Desktop\图书馆\有附件的书目筛选结果.xlsx


In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# 文件路径
input_file = r".\有附件的书目筛选结果.xlsx"
output_file = r".\有附件的书目分类与高亮结果.xlsx"

# 1. 定义分类关键词字典 (可根据实际情况随时增补)
rules = {
    '晚辈/门生': ['学生', '受业', '后学', '晚生', '末学', '恩师', '拜赠', '敬呈', '门生', '弟子', '师'],
    '同仁/平辈': ['兄', '老友', '学长', '同志', '雅正', '指正', '教正', '惠存', '大雅', '方家', '斧正', '先生雅正', '教授', '所长', '敬赠'],
    '师长/长辈': ['祖武存', '祖武读', '贤弟', '小友', '世侄', '祖武留念'],
    '官方/机构': ['委员会', '大学', '研究所', '基金会', '图书馆', '出版社', '编辑部', '公章', '印章', '赠书']
}

def classify_relation(text):
    if pd.isna(text):
        return "待核对"
    text = str(text)
    
    # 遍历规则进行匹配
    for category, keywords in rules.items():
        for kw in keywords:
            if kw in text:
                return category
    
    # 无明显特征词，留给人工核对
    return "待核对"

try:
    print("正在进行自动分类...")
    df = pd.read_excel(input_file)
    
    # 清理列名以防万一
    df.columns = df.columns.str.replace('\n', '').str.replace('\r', '').str.strip()
    
    # 把附件形式和内容结合起来作为判断依据
    content_col = '附件内容'
    form_col = '附件形式（手写、信件、卡片等）'
    
    # 如果某列不存在，用空字符串代替防止报错
    if content_col not in df.columns: df[content_col] = ''
    if form_col not in df.columns: df[form_col] = ''
        
    df['综合文本'] = df[content_col].fillna('') + df[form_col].fillna('')
    
    # 应用分类函数，生成新列
    df['关系分类'] = df['综合文本'].apply(classify_relation)
    df.drop(columns=['综合文本'], inplace=True) # 删掉辅助列
    
    # 先导出为Excel
    df.to_excel(output_file, index=False)
    
    # 2. 用 openpyxl 给“待核对”的单元格上色
    print("正在为无法识别的数据标记颜色...")
    wb = load_workbook(output_file)
    ws = wb.active
    
    # 查找“关系分类”所在的列号
    col_idx = None
    for col in range(1, ws.max_column + 1):
        if ws.cell(row=1, column=col).value == '关系分类':
            col_idx = col
            break
            
    # 定义亮黄色填充
    yellow_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
    
    unclassified_count = 0
    if col_idx:
        for row in range(2, ws.max_row + 1):
            cell = ws.cell(row=row, column=col_idx)
            if cell.value == "待核对":
                cell.fill = yellow_fill
                unclassified_count += 1
                
    wb.save(output_file)
    print(f"\n--- 处理完成！ ---")
    print(f"总数据量：{len(df)} 行")
    print(f"需要人工矫正的标黄数据：{unclassified_count} 行")
    print(f"文件已保存至：{output_file}")

except Exception as e:
    print(f"运行出错：{e}")

正在进行自动分类...
正在为无法识别的数据标记颜色...

--- 处理完成！ ---
总数据量：1262 行
需要人工矫正的标黄数据：679 行
文件已保存至：E:\Library\图书馆\有附件的书目分类与高亮结果.xlsx


In [ ]:
import pandas as pd
import re
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# 文件路径
input_file = r".\有附件的书目筛选结果.xlsx"
output_file = r".\有附件的书目分类与高亮结果.xlsx"

# 1. 定义分类关键词字典 (已将同仁/平辈改为“有朋”)
rules = {
    '晚辈/门生': ['学生', '受业', '后学', '晚生', '末学', '恩师', '拜赠', '敬呈', '门生', '弟子', '师'],
    '友朋': ['兄', '老友', '学长', '同志', '雅正', '指正', '教正', '惠存', '大雅', '方家', '斧正', '先生雅正', '教授', '所长', '敬赠', '友朋'],
    '师长/长辈': ['祖武存', '祖武读', '贤弟', '小友', '世侄', '祖武留念'],
    '官方/机构': ['委员会', '大学', '研究所', '基金会', '图书馆', '出版社', '编辑部', '公章', '印章', '赠书']
}

# 辅助函数：利用正则表达式从“著者简介”中提取出生年份
def extract_birth_year(intro_text):
    if pd.isna(intro_text):
        return None
    text = str(intro_text)
    # 匹配常见的年份格式，例如 "1963年"、"(1921-"、"生于1945"
    match = re.search(r'(18\d{2}|19\d{2})(?:年|[-~—生])', text)
    if match:
        return int(match.group(1))
    return None

# 核心分类函数：结合文本关键词和年龄判断
def classify_relation(row):
    text = str(row.get('综合文本', ''))
    intro = row.get('著者简介', '')
    
    # 第一步：如果是明显的机构赠书，优先判断为机构
    for kw in rules['官方/机构']:
        if kw in text:
            return '官方/机构'
            
    # 第二步：根据明显的文本称呼词进行分类
    for category in ['晚辈/门生', '友朋', '师长/长辈']:
        for kw in rules[category]:
            if kw in text:
                return category
                
    # 第三步：如果题签用词不清晰，尝试提取出生年份进行辅助判断
    birth_year = extract_birth_year(intro)
    if birth_year:
        # 陈祖武先生生于1943年
        if birth_year <= 1933:
            return '师长/长辈'
        elif 1934 <= birth_year <= 1953:
            return '友朋'
        else:
            return '晚辈/门生'
            
    # 既没有特征词，也没有提取到生年，留给人工核对
    return "待核对"

try:
    print("正在进行结合年龄的智能分类...")
    df = pd.read_excel(input_file)
    
    # 清理列名以防万一
    df.columns = df.columns.str.replace('\n', '').str.replace('\r', '').str.strip()
    
    # 生成综合文本
    content_col = '附件内容'
    form_col = '附件形式（手写、信件、卡片等）'
    if content_col not in df.columns: df[content_col] = ''
    if form_col not in df.columns: df[form_col] = ''
    if '著者简介' not in df.columns: df['著者简介'] = ''
        
    df['综合文本'] = df[content_col].fillna('') + df[form_col].fillna('')
    
    # 应用分类函数（这里改成了按行应用 axis=1，因为我们需要同时用到文本和简介两列的数据）
    df['关系分类'] = df.apply(classify_relation, axis=1)
    df.drop(columns=['综合文本'], inplace=True) 
    
    df.to_excel(output_file, index=False)
    
    # 用 openpyxl 给“待核对”的单元格上色
    print("正在为无法识别的数据标记颜色...")
    wb = load_workbook(output_file)
    ws = wb.active
    
    col_idx = None
    for col in range(1, ws.max_column + 1):
        if ws.cell(row=1, column=col).value == '关系分类':
            col_idx = col
            break
            
    yellow_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
    
    unclassified_count = 0
    if col_idx:
        for row in range(2, ws.max_row + 1):
            cell = ws.cell(row=row, column=col_idx)
            if cell.value == "待核对":
                cell.fill = yellow_fill
                unclassified_count += 1
                
    wb.save(output_file)
    print(f"\n--- 处理完成！ ---")
    print(f"总数据量：{len(df)} 行")
    print(f"未能提取到年份且用词不明，需人工矫正：{unclassified_count} 行")
    print(f"文件已保存至：{output_file}")

except Exception as e:
    print(f"运行出错：{e}")

正在进行结合年龄的智能分类...
正在为无法识别的数据标记颜色...

--- 处理完成！ ---
总数据量：1262 行
未能提取到年份且用词不明，需人工矫正：465 行
文件已保存至：E:\Library\图书馆\有附件的书目分类与高亮结果.xlsx


In [ ]:
import pandas as pd
import re
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# 1. 定义原文件路径和新文件的保存路径（我在文件名后加了_新增送书人，以免覆盖你之前辛苦跑出的数据）
input_file = r".\有附件的书目分类与高亮结果.xlsx"
output_file = r".\有附件的书目分类与高亮结果_新增送书人.xlsx"

# 辅助函数：提取送书人名字
def extract_sender(intro_text):
    if pd.isna(intro_text):
        return ""
    
    # 转换为字符串并去掉首尾多余的空白字符
    text = str(intro_text).strip()
    if not text:
        return ""
    
    # 按照换行符分段，并过滤掉空行
    paragraphs = [p.strip() for p in text.split('\n') if p.strip()]
    if not paragraphs:
        return ""
    
    # 获取最后一段
    last_paragraph = paragraphs[-1]
    
    # 提取名字：遇到中文逗号、英文逗号、左右括号、空格等标点符号就切断，取最前面的部分
    parts = re.split(r'[，,（( \t。；;：:]', last_paragraph)
    name = parts[0]
    
    # 如果提取出的名字异常长（超过10个字），可能提取不准，加个省略号提示人工核对
    if len(name) > 10:
        return name[:10] + "..."
        
    return name

try:
    print("正在读取数据...")
    df = pd.read_excel(input_file)
    
    # 清洗列名，防止隐藏换行符干扰
    df.columns = df.columns.str.replace('\n', '').str.replace('\r', '').str.strip()
    
    # 确保有“著者简介”这一列
    if '著者简介' not in df.columns:
        print("警告：未找到‘著者简介’列，送书人将全部为空。")
        df['著者简介'] = ''
        
    print("正在从‘著者简介’最后一段提取送书人名字...")
    # 对整列应用提取函数
    senders = df['著者简介'].apply(extract_sender)
    
    # 找到“关系分类”列的位置，在其前面插入“送书人”列
    if '关系分类' in df.columns:
        col_idx = df.columns.get_loc('关系分类')
        # 如果不小心已经有了“送书人”列，先删掉以防报错
        if '送书人' in df.columns:
            df.drop(columns=['送书人'], inplace=True)
            col_idx = df.columns.get_loc('关系分类')
            
        # 插入列（位置索引，列名，数据）
        df.insert(col_idx, '送书人', senders)
    else:
        # 如果万一没找到关系分类，就直接加在表格最后
        df['送书人'] = senders
        
    # 保存结果，先不加颜色
    df.to_excel(output_file, index=False)
    
    # ------------------ 下面是恢复黄色高亮的代码 ------------------
    print("正在恢复‘待核对’数据的黄色高亮标记...")
    wb = load_workbook(output_file)
    ws = wb.active
    
    # 重新定位“关系分类”所在列（因为前面插入了新列，索引变了）
    relation_col_idx = None
    for col in range(1, ws.max_column + 1):
        if ws.cell(row=1, column=col).value == '关系分类':
            relation_col_idx = col
            break
            
    yellow_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
    
    if relation_col_idx:
        for row in range(2, ws.max_row + 1):
            cell = ws.cell(row=row, column=relation_col_idx)
            if cell.value == "待核对":
                cell.fill = yellow_fill
                
    wb.save(output_file)
    print(f"\n--- 处理完成！ ---")
    print(f"文件已保存至：{output_file}")

except Exception as e:
    print(f"运行出错：{e}")

正在读取数据...
正在从‘著者简介’最后一段提取送书人名字...
正在恢复‘待核对’数据的黄色高亮标记...

--- 处理完成！ ---
文件已保存至：E:\Library\图书馆\有附件的书目分类与高亮结果_新增送书人.xlsx


In [ ]:
import pandas as pd
import re
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# 文件路径
input_file = r".\有附件的书目分类与高亮结果_新增送书人.xlsx"
output_file = r".\有附件的书目分类与高亮结果_著者简介精准提取.xlsx"

# 精准提取核心简介的函数
def extract_core_bios(text):
    if pd.isna(text):
        return text
    
    text = str(text).strip()
    if not text:
        return text
        
    # 按换行符分割成段落，去掉空行
    paragraphs = [p.strip() for p in text.split('\n') if p.strip()]
    kept_paragraphs = []
    
    # 核心正则匹配规则：在段落的前20个字符内，寻找以下特征：
    # 1. 包含括号+年份，例如：范军（1968—） 或 (1975-)
    # 2. 包含逗号+性别，例如：张三，男， 或 李四, 女,
    # 3. 包含逗号+出生年，例如：王五，1980年生
    pattern = re.compile(r'^.{0,20}(?:[（(]\s*(?:18|19|20)\d{2}|[，,]\s*[男女]\s*[，,。]|[，,]\s*(?:18|19|20)\d{2}\s*年)')
    
    for p in paragraphs:
        if pattern.search(p):
            kept_paragraphs.append(p)
            
    # 如果找到了符合条件的段落，就用换行符把它们重新拼起来
    if kept_paragraphs:
        return '\n'.join(kept_paragraphs)
        
    # 如果该单元格里没有任何一段符合上述特征，为了安全起见保留原样，防止误删
    return text

try:
    print("正在读取数据...")
    df = pd.read_excel(input_file)
    
    # 清理列名
    df.columns = df.columns.str.replace('\n', '').str.replace('\r', '').str.strip()
    
    if '著者简介' in df.columns:
        # Excel的第577行到610行，在pandas里的索引对应是 575 到 608
        start_idx = 577 - 2
        end_idx = 610 - 2
        
        start_idx = max(0, start_idx)
        end_idx = min(len(df) - 1, end_idx)
        
        print(f"正在精准提取 Excel 第 577 行到第 610 行的核心著者简介...")
        
        # 遍历指定的行区间并修改
        for i in range(start_idx, end_idx + 1):
            original_text = df.at[i, '著者简介']
            df.at[i, '著者简介'] = extract_core_bios(original_text)
            
        print("文本精准截取完成！")
    else:
        print("警告：未找到‘著者简介’列！")

    # 保存新文件
    df.to_excel(output_file, index=False)
    
    # --- 恢复黄色高亮 ---
    print("正在恢复‘待核对’数据的黄色高亮标记...")
    wb = load_workbook(output_file)
    ws = wb.active
    
    relation_col_idx = None
    for col in range(1, ws.max_column + 1):
        if ws.cell(row=1, column=col).value == '关系分类':
            relation_col_idx = col
            break
            
    yellow_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
    
    if relation_col_idx:
        for row in range(2, ws.max_row + 1):
            cell = ws.cell(row=row, column=relation_col_idx)
            if cell.value == "待核对":
                cell.fill = yellow_fill
                
    wb.save(output_file)
    print(f"\n--- 处理完成！ ---")
    print(f"文件已成功保存至：{output_file}")

except PermissionError:
    print("错误：文件可能在 Excel 中被打开了。请关闭后再运行代码。")
except Exception as e:
    print(f"运行出错：{e}")

正在读取数据...
正在精准提取 Excel 第 577 行到第 610 行的核心著者简介...
文本精准截取完成！
正在恢复‘待核对’数据的黄色高亮标记...

--- 处理完成！ ---
文件已成功保存至：E:\Library\图书馆\有附件的书目分类与高亮结果_著者简介精准提取.xlsx


In [ ]:
import pandas as pd
import re
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# 1. 定义输入和输出路径
# 输入文件：刚刚经过精准提取核心简介的文件
input_file = r".\有附件的书目分类与高亮结果_著者简介精准提取.xlsx"
# 输出文件：刷新送书人后的最终文件
output_file = r".\有附件的书目分类与高亮结果_最终版.xlsx"

# 辅助函数：提取送书人名字
def extract_sender(intro_text):
    if pd.isna(intro_text):
        return ""
    
    # 转换为字符串并去掉首尾多余的空白字符
    text = str(intro_text).strip()
    if not text:
        return ""
    
    # 按照换行符分段，并过滤掉空行
    paragraphs = [p.strip() for p in text.split('\n') if p.strip()]
    if not paragraphs:
        return ""
    
    # 获取最后一段
    last_paragraph = paragraphs[-1]
    
    # 提取名字：遇到中文逗号、英文逗号、左右括号、空格等标点符号就切断，取最前面的部分
    parts = re.split(r'[，,（( \t。；;：:]', last_paragraph)
    name = parts[0]
    
    # 如果提取出的名字异常长（超过10个字），可能提取不准，加个省略号提示人工核对
    if len(name) > 10:
        return name[:10] + "..."
        
    return name

try:
    print("正在读取数据...")
    df = pd.read_excel(input_file)
    
    # 清洗列名，防止隐藏换行符干扰
    df.columns = df.columns.str.replace('\n', '').str.replace('\r', '').str.strip()
    
    if '著者简介' not in df.columns:
        print("警告：未找到‘著者简介’列，送书人将全部为空。")
        df['著者简介'] = ''
        
    print("正在基于精准提取后的‘著者简介’重新刷新送书人名字...")
    # 对整列应用提取函数
    senders = df['著者简介'].apply(extract_sender)
    
    # 检查是否已经存在“送书人”列
    if '送书人' in df.columns:
        # 如果存在，直接覆盖更新数据
        df['送书人'] = senders
    elif '关系分类' in df.columns:
        # 如果不存在，找到“关系分类”列的位置，在其前面插入“送书人”列
        col_idx = df.columns.get_loc('关系分类')
        df.insert(col_idx, '送书人', senders)
    else:
        # 如果万一没找到关系分类，就直接加在表格最后
        df['送书人'] = senders
        
    # 保存结果，先不加颜色
    df.to_excel(output_file, index=False)
    
    # ------------------ 下面是恢复黄色高亮的代码 ------------------
    print("正在恢复‘待核对’数据的黄色高亮标记...")
    wb = load_workbook(output_file)
    ws = wb.active
    
    # 定位“关系分类”所在列
    relation_col_idx = None
    for col in range(1, ws.max_column + 1):
        if ws.cell(row=1, column=col).value == '关系分类':
            relation_col_idx = col
            break
            
    yellow_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
    
    if relation_col_idx:
        for row in range(2, ws.max_row + 1):
            cell = ws.cell(row=row, column=relation_col_idx)
            if cell.value == "待核对":
                cell.fill = yellow_fill
                
    wb.save(output_file)
    print(f"\n--- 处理完成！ ---")
    print(f"新的送书人数据已刷新，文件已保存至：{output_file}")

except PermissionError:
    print("\n错误：权限被拒绝。请确保你没有在Excel里打开要写入的文件。如果打开了，请关闭后再运行！")
except Exception as e:
    print(f"\n运行出错：{e}")

正在读取数据...
正在基于精准提取后的‘著者简介’重新刷新送书人名字...
正在恢复‘待核对’数据的黄色高亮标记...

--- 处理完成！ ---
新的送书人数据已刷新，文件已保存至：E:\Library\图书馆\有附件的书目分类与高亮结果_最终版.xlsx


In [4]:
import pandas as pd
import math
from pyvis.network import Network

# 1. 读取数据并清洗
file_path = r".\赠阅图书整理_时间已提取.xlsx"
df = pd.read_excel(file_path)

# 【优化】：为了防止报错，加个.copy()
df_valid = df.dropna(subset=['送书人', '关系分类']).copy()
df_valid.loc[:, '送书人'] = df_valid['送书人'].astype(str).str.strip()
df_valid.loc[:, '关系分类'] = df_valid['关系分类'].astype(str).str.strip()

# ==========================================
# 【关键修改 1】：处理多人捐赠（用顿号隔开）的情况
# ==========================================
# 将含有“、”的字符串拆分成列表，比如 ["张三", "李四"]
df_valid['送书人'] = df_valid['送书人'].str.split('、')
# explode 函数会把列表炸开，自动变成多行数据，每人单独算一次！
df_valid = df_valid.explode('送书人')
# 再次清理拆分后可能多出来的首尾空格
df_valid['送书人'] = df_valid['送书人'].str.strip()

# 统计每个人最终的赠书总册数
sender_counts = df_valid.groupby(['送书人', '关系分类']).size().reset_index(name='册数')

# 2. 定义行星轨道的半径（多圈）和颜色
orbit_config = {
    "师长/长辈": {"radii": [100], "color": "#FF9900", "nodes": []},
    "官方/机构": {"radii": [200], "color": "#990099", "nodes": []},
    "晚辈/门生": {"radii": [300, 400], "color": "#109618", "nodes": []},             
    "友朋":     {"radii": [500, 600, 700, 800], "color": "#3366CC", "nodes": []}, 
    "待核对":   {"radii": [900], "color": "#999999", "nodes": []}
}

# 把数据装入对应的分类
for _, row in sender_counts.iterrows():
    sender = row['送书人']
    relation = row['关系分类']
    count = row['册数']
    
    if sender == "" or sender == "nan" or sender == "陈祖武": continue
        
    if relation in orbit_config:
        orbit_config[relation]["nodes"].append({"name": sender, "count": count})
    else:
        orbit_config["待核对"]["nodes"].append({"name": sender, "count": count})

# 3. 初始化网络图（浅色背景）
net = Network(height='800px', width='100%', bgcolor='#f8f9fa', font_color='#333333', directed=False, cdn_resources='remote')

# 彻底关闭全局物理引擎
net.toggle_physics(False)

# 4. 放置中心恒星：陈祖武先生（锁定在坐标 x=0, y=0）
net.add_node("陈祖武", size=45, color='#c23531', title="核心: 陈祖武先生", label="陈祖武", x=0, y=0, physics=False)

# 5. 利用三角函数，把学者均匀排列在多圈轨道上
for relation, config in orbit_config.items():
    nodes_list = config["nodes"]
    n_nodes = len(nodes_list)
    radii = config["radii"]
    num_circles = len(radii)
    color = config["color"]
    
    # 遍历当前分类的每一圈轨道
    for c_idx, R in enumerate(radii):
        # 将该分类下的节点尽可能均匀地切分，分配到每一圈
        start_idx = c_idx * n_nodes // num_circles
        end_idx = (c_idx + 1) * n_nodes // num_circles
        circle_nodes = nodes_list[start_idx:end_idx]
        n_circle_nodes = len(circle_nodes)
        
        # 将分配到这一圈的节点均匀画在圆周上
        for i, node_data in enumerate(circle_nodes):
            sender = node_data["name"]
            count = node_data["count"]
            
            # ==========================================
            # 【关键修改 2】：统一所有周围节点的大小为固定值
            # ==========================================
            node_size = 15 
            
            # 纯数学计算坐标
            angle = 2 * math.pi * i / n_circle_nodes if n_circle_nodes > 0 else 0
            
            # 视觉优化：如果是偶数圈，让节点错开半个身位
            if c_idx % 2 == 1 and n_circle_nodes > 0:
                angle += (math.pi / n_circle_nodes)
                
            x = R * math.cos(angle)
            y = R * math.sin(angle)
            
            # ==========================================
            # 【关键修改 3】：在标签后添加冒号和数量
            # ==========================================
            # label 属性控制显示在圆圈下方/里面的文字
            label_text = f"{sender} {count}本"
            # title 控制鼠标悬停时的提示气泡
            title_text = f"学者: {sender}\n分类: {relation}\n赠书: {count} 册"
            
            # 强行写入 x, y 坐标，并指定 physics=False
            net.add_node(sender, size=node_size, color=color, title=title_text, label=label_text, x=x, y=y, physics=False)
            net.add_edge("陈祖武", sender, color="#e0e0e0")

# 6. 保存网页
static_html = r".\陈祖武交游图谱_同心圆纯静态版.html"
net.save_graph(static_html)

print(f"--- 处理完成！ ---")
print(f"包含合著拆分与直观数量标签的图谱已生成：{static_html}")

--- 处理完成！ ---
包含合著拆分与直观数量标签的图谱已生成：.\陈祖武交游图谱_同心圆纯静态版.html


In [ ]:
import pandas as pd
import numpy as np

# 1. 配置文件路径
file_source = r".\赠阅图书整理_时间已提取.xlsx"  # 核对好的文件
file_target = r".\陈祖武先生捐赠书籍整理书目.xlsx" # 目标总表
file_output = r".\陈祖武先生捐赠书籍整理书目_最终版_含赠阅标签.xlsx" # 生成的新表

try:
    print("正在读取两个表格的数据...")
    df_source = pd.read_excel(file_source)
    df_target = pd.read_excel(file_target)

    # 深度清理列名
    df_source.columns = df_source.columns.str.replace(r'[\n\r]', '', regex=True).str.strip()
    df_target.columns = df_target.columns.str.replace(r'[\n\r]', '', regex=True).str.strip()

    print("正在执行精准覆盖，并为“类型”列添加‘赠阅’后缀...")
    
    if '序号' not in df_source.columns or '序号' not in df_target.columns:
        print("致命错误：两张表中必须都包含 '序号' 列作为匹配基准！")
    else:
        # 将“序号”设为索引进行对齐
        df_target.set_index('序号', inplace=True)
        df_source.set_index('序号', inplace=True)

        # ----------------------------------------------------
        # 任务 1：准备同名列的更新数据
        # ----------------------------------------------------
        cols_to_sync = ['附件形式（手写、信件、卡片等）', '附件内容', '著者简介']
        for col in cols_to_sync:
            if col not in df_target.columns:
                df_target[col] = np.nan
        
        update_data = df_source[cols_to_sync].copy()

        # ----------------------------------------------------
        # 任务 2：将“关系分类”加上“赠阅”二字，映射到“类型”
        # ----------------------------------------------------
        if '关系分类' in df_source.columns:
            if '类型' not in df_target.columns:
                df_target['类型'] = np.nan
            
            # 处理逻辑：如果关系分类不为空，则在末尾加上“赠阅”
            update_data['类型'] = df_source['关系分类'].apply(
                lambda x: f"{str(x).strip()}赠阅" if pd.notna(x) and str(x).strip() != "" else x
            )
        else:
            print("警告：未找到‘关系分类’列，无法更新类型。")

        # ----------------------------------------------------
        # 核心执行：数据精准覆盖
        # ----------------------------------------------------
        df_target.update(update_data)

        # 恢复表格结构
        df_target.reset_index(inplace=True)

        # 保存结果
        df_target.to_excel(file_output, index=False)
        
        print(f"\n--- 处理完成！ ---")
        print(f"1. [附件形式]、[附件内容]、[著者简介] 已同步。")
        print(f"2. [类型] 列已根据关系分类更新，并统一添加了‘赠阅’后缀。")
        print(f"新文件位置：{file_output}")

except FileNotFoundError:
    print("\n错误：找不到文件，请检查路径。")
except PermissionError:
    print("\n错误：请关闭相关的 Excel 文件后再运行代码！")
except Exception as e:
    print(f"\n运行出错：{e}")

正在读取两个表格的数据...


d:\Anaconda\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


正在执行精准覆盖，并为“类型”列添加‘赠阅’后缀...

--- 处理完成！ ---
1. [附件形式]、[附件内容]、[著者简介] 已同步。
2. [类型] 列已根据关系分类更新，并统一添加了‘赠阅’后缀。
新文件位置：E:\Library\图书馆\陈祖武先生捐赠书籍整理书目_最终版_含赠阅标签.xlsx


In [ ]:
import pandas as pd
import re

# 1. 配置文件路径
file_input = r".\赠阅图书整理+时间版本.xlsx"
file_output = r".\赠阅图书整理_时间已提取.xlsx"

# 2. 定义时间提取函数（核心魔法）
def extract_time_from_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    
    # 定义匹配规则（按优先级从精细到粗略排序）
    # 规则1：标准年月日 (例：2004.12.23, 2008年1月19日, 2008-01-19)
    p1 = r'((?:19|20)\d{2}\s*[年\.\-]\s*\d{1,2}\s*[月\.\-]\s*\d{1,2}\s*日?)'
    
    # 规则2：中文数字年月日 (例：二00八.一.二0, 二〇一六年八月十日)
    p2 = r'([一二三四五六七八九〇0]{4}\s*[年\.\-]\s*[一二三四五六七八九十〇0]{1,3}\s*[月\.\-]\s*[一二三四五六七八九十〇0]{1,3}\s*日?)'
    
    # 规则3：标准年月 (例：2016.08, 2016年8月)
    p3 = r'((?:19|20)\d{2}\s*[年\.\-]\s*\d{1,2}\s*月?)'
    
    # 规则4：中文年月 (例：二0一六.八, 二〇一六年八月)
    p4 = r'([一二三四五六七八九〇0]{4}\s*[年\.\-]\s*[一二三四五六七八九十〇0]{1,3}\s*月?)'
    
    # 规则5：年份+文人雅称/季节 (例：二00二年晚秋, 2002年春)
    p5 = r'([一二三四五六七八九〇00-9]{4}\s*年\s*(?:春|夏|秋|冬|初春|仲夏|晚秋|深冬|孟春|初冬|岁首|岁末)?)'
    
    patterns = [p1, p2, p3, p4, p5]
    
    # 逐个规则去扫描文字
    for p in patterns:
        match = re.search(p, text)
        if match:
            return match.group(1).strip() # 找到就立刻返回提取出的时间
            
    return "" # 如果所有规则都没找到，返回空值，留待人工核对

try:
    print("正在读取数据...")
    df = pd.read_excel(file_input)
    
    # 清理列名，防止隐藏换行符
    df.columns = df.columns.str.replace(r'[\n\r]', '', regex=True).str.strip()
    
    if '附件内容' not in df.columns:
        print("错误：表格中没有找到‘附件内容’这一列！")
    else:
        # 如果没有‘时间’列，就新建一列
        if '时间' not in df.columns:
            # 找到‘送书人’的位置，插在它后面
            if '送书人' in df.columns:
                idx = df.columns.get_loc('送书人') + 1
                df.insert(idx, '时间', "")
            else:
                df['时间'] = ""
                
        print("正在运用正则引擎扫描并提取赠阅时间...")
        
        # 将提取函数应用到整列数据
        df['时间'] = df['附件内容'].apply(extract_time_from_text)
        
        # 统计提取成功了多少条
        success_count = (df['时间'] != "").sum()
        
        # 保存为新文件
        df.to_excel(file_output, index=False)
        
        print(f"\n--- 提取完成！ ---")
        print(f"共成功从附件内容中提取出 {success_count} 条时间记录。")
        print(f"新的文件已保存至：{file_output}")

except PermissionError:
    print("错误：文件可能正在被 Excel 打开。请关闭 Excel 文件后再运行！")
except Exception as e:
    print(f"运行出错：{e}")

正在读取数据...
正在运用正则引擎扫描并提取赠阅时间...

--- 提取完成！ ---
共成功从附件内容中提取出 344 条时间记录。
新的文件已保存至：E:\Library\图书馆\赠阅图书整理_时间已提取.xlsx


In [7]:
import pandas as pd
import json

# 1. 配置文件路径
file_input = r"E:\Library\赠阅图书整理_时间已提取.xlsx"
file_output = r"E:\Library\陈祖武先生受赠图书明细_含附件内容.html"

try:
    print("正在读取数据并构建大屏...")
    df = pd.read_excel(file_input)
    
    # 2. 数据清洗
    df = df.dropna(subset=['年', '关系分类'])
    df['年'] = pd.to_numeric(df['年'], errors='coerce')
    df = df.dropna(subset=['年'])
    df = df[(df['年'] >= 1950) & (df['年'] <= 2050)]
    df['年'] = df['年'].astype(int)
    
    # 智能合成标准化日期
    def synthesize_clean_date(row):
        y = f"{int(row['年'])}年"
        m_val = row.get('月', pd.NA)
        m = ""
        if pd.notna(m_val) and str(m_val).strip() != "":
            try:
                m = f"{int(float(str(m_val)))}月"
            except ValueError:
                m = f"{str(m_val).strip()}月"
                
        d_val = row.get('日', pd.NA)
        d = ""
        if pd.notna(d_val) and str(d_val).strip() != "":
            try:
                d = f"{int(float(str(d_val)))}日"
            except ValueError:
                d = f"{str(d_val).strip()}日"
                
        return f"{y}{m}{d}"

    df['标准化时间'] = df.apply(synthesize_clean_date, axis=1)
    
    # 3. 准备图表数据
    relations = ["师长/长辈", "友朋", "晚辈/门生", "官方/机构", "待核对"]
    color_dict = {
        "师长/长辈": "#FF9900", 
        "友朋": "#3366CC",      
        "晚辈/门生": "#109618",   
        "官方/机构": "#990099",
        "待核对": "#999999"
    }

    min_year = df['年'].min()
    max_year = df['年'].max()
    years = list(range(int(min_year), int(max_year) + 1))
    years_str = [str(y) for y in years]
    
    series_list = []
    total_counts = [0] * len(years)
    
    for rel in relations:
        rel_data = []
        for i, year in enumerate(years):
            count = len(df[(df['年'] == year) & (df['关系分类'] == rel)])
            rel_data.append(count)
            total_counts[i] += count
            
        series_list.append({
            "name": rel,
            "type": 'bar',
            "stack": 'Total',
            "itemStyle": {"color": color_dict.get(rel, "#999999")},
            "data": rel_data
        })
        
    series_list.append({
        "name": '年度总册数',
        "type": 'line',
        "smooth": True,
        "symbolSize": 8,
        "itemStyle": {"color": "#333333"},
        "lineStyle": {"width": 3, "type": "dashed"},
        "data": total_counts
    })
    
    # 4. 准备底部列表数据
    table_rows = ""
    df_sorted = df.sort_values(by=['年', '月', '日'], ascending=[True, True, True], na_position='first')
    
    for _, row in df_sorted.iterrows():
        year = str(row['年'])
        relation = str(row['关系分类']).strip() if pd.notna(row['关系分类']) else ""
        sender = str(row['送书人']).strip() if pd.notna(row['送书人']) else ""
        title = str(row['题名']).strip() if pd.notna(row['题名']) else ""
        time_str = str(row['标准化时间'])
        
        # 【新增提取代码】：安全地提取“附件内容”列
        attachment = str(row['附件内容']).strip() if '附件内容' in row and pd.notna(row['附件内容']) else ""
        
        # 【修改 HTML 拼接】：把附件内容加到最后面
        table_rows += f"<tr><td>{year}</td><td>{relation}</td><td>{sender}</td><td>{title}</td><td>{time_str}</td><td>{attachment}</td></tr>\n"

    # 5. HTML 前端代码模板
    html_template = """
    <!DOCTYPE html>
    <html lang="zh-CN">
    <head>
        <meta charset="utf-8">
        <title>陈祖武先生受赠图书明细</title>
        <script src="https://cdnjs.cloudflare.com/ajax/libs/echarts/5.4.2/echarts.min.js"></script>
        <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
        <script src="https://code.jquery.com/jquery-3.6.0.min.js"></script>
        <script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
        
        <style>
            body { font-family: 'Microsoft YaHei', sans-serif; background-color: #f0f2f5; margin: 0; padding: 30px; }
            .container { width: 95%; max-width: 1600px; margin: 0 auto; background: white; padding: 40px; box-shadow: 0 4px 12px rgba(0,0,0,0.05); border-radius: 12px; }
            h1 { text-align: center; color: #2c3e50; margin-bottom: 40px; font-size: 28px; }
            #chart { width: 100%; height: 550px; margin-bottom: 50px; cursor: pointer; }
            .table-container { border-top: 2px dashed #eee; padding-top: 30px; }
            h2 { text-align: center; color: #34495e; margin-bottom: 20px; font-size: 22px; }
            table.dataTable { border-collapse: collapse; width: 100%; }
            table.dataTable thead th { background-color: #f8f9fa; color: #333; }
            
            /* 适度控制一下附件内容列的宽度，防止文字太长撑爆屏幕 */
            table.dataTable td:nth-child(6) { max-width: 400px; white-space: normal; word-wrap: break-word; line-height: 1.5; color: #555; }
            
            .interaction-board {
                display: flex;
                justify-content: space-between;
                align-items: center;
                background: #f8f9fa;
                padding: 15px 25px;
                border-radius: 8px;
                border: 1px solid #e9ecef;
                border-left: 5px solid #3b6291;
                margin-bottom: 20px;
            }
            .status-text { font-size: 16px; color: #2c3e50; }
            .highlight-year { color: #d94e5d; font-weight: bold; font-size: 18px; margin: 0 5px; }
            
            #resetBtn {
                background-color: #3b6291; color: white; border: none; padding: 8px 20px;
                border-radius: 4px; cursor: pointer; font-size: 14px; font-weight: bold;
                transition: all 0.3s; display: none; 
            }
            #resetBtn:hover { background-color: #2c4a6e; box-shadow: 0 2px 5px rgba(0,0,0,0.2); }
            
            .dataTables_filter { display: none; }
            .global-search { 
                padding: 8px 12px; border: 1px solid #ced4da; border-radius: 4px; 
                outline: none; width: 220px; transition: border-color 0.3s;
            }
            .global-search:focus { border-color: #3b6291; }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>陈祖武先生受赠图书明细</h1>
            <div id="chart" title="点击柱子查看当年明细"></div>
            
            <div class="table-container">
                <h2>赠阅明细检索</h2>
                
                <div class="interaction-board">
                    <div class="status-text">
                        <span>💡 <strong>操作提示：</strong>点击上方图表中的年份柱子即可过滤。</span>
                        <span style="margin-left:20px;">当前查看：<span id="currentYearDisplay" class="highlight-year">所有年份</span></span>
                    </div>
                    <div>
                        <input type="text" id="globalSearch" class="global-search" placeholder="🔍 检索书名/人名/题词...">
                        <button id="resetBtn">🔄 显示全部年份</button>
                    </div>
                </div>
                
                <table id="myTable" class="display">
                    <thead>
                        <tr>
                            <th>年份</th>
                            <th>关系分类</th>
                            <th>送书人</th>
                            <th>赠书书名</th>
                            <th>规范赠阅时间</th>
                            <th>附件内容</th>
                        </tr>
                    </thead>
                    <tbody>
                        {{TABLE_ROWS}}
                    </tbody>
                </table>
            </div>
        </div>

        <script>
            var chartDom = document.getElementById('chart');
            var myChart = echarts.init(chartDom);
            
            var option = {
                tooltip: { trigger: 'axis', axisPointer: { type: 'cross' } },
                legend: { top: 'bottom' },
                grid: { left: '3%', right: '4%', bottom: '10%', containLabel: true },
                dataZoom: [ { type: 'inside' }, { type: 'slider', bottom: '0%' } ],
                xAxis: { type: 'category', data: {{YEARS}}, axisLabel: { rotate: 45 } },
                yAxis: { type: 'value', name: '赠书册数 (本)' },
                series: {{SERIES_DATA}}
            };
            myChart.setOption(option);

            $(document).ready(function() {
                if ($.fn.dataTable.ext && $.fn.dataTable.ext.search) {
                    $.fn.dataTable.ext.search = [];
                }
                
                var table = $('#myTable').DataTable({
                    "order": [[ 0, "asc" ]], 
                    "pageLength": 10,
                    "language": { 
                        "processing": "处理中...",
                        "lengthMenu": "显示 _MENU_ 项结果",
                        "zeroRecords": "📭 该年份暂无赠阅记录哦~",
                        "info": "显示第 _START_ 至 _END_ 项结果，共 _TOTAL_ 项",
                        "infoEmpty": "显示第 0 至 0 项结果，共 0 项",
                        "infoFiltered": "(由 _MAX_ 项结果过滤)",
                        "search": "搜索:",
                        "emptyTable": "表中数据为空",
                        "paginate": { "first": "首页", "previous": "上页", "next": "下页", "last": "末页" }
                    }
                });
                
                myChart.on('click', function (params) {
                    var clickYear = params.name; 
                    if (clickYear) {
                        table.column(0).search('^' + clickYear + '$', true, false).draw();
                        
                        $('#currentYearDisplay').text(clickYear + '年');
                        $('#resetBtn').fadeIn(); 
                        
                        $('html, body').animate({
                            scrollTop: $(".table-container").offset().top - 20
                        }, 500);
                    }
                });
                
                $('#resetBtn').on('click', function() {
                    table.column(0).search('').draw(); 
                    $('#currentYearDisplay').text('所有年份');
                    $(this).fadeOut(); 
                });
                
                function debounce(func, wait) {
                    var timeout;
                    return function() {
                        var context = this, args = arguments;
                        clearTimeout(timeout);
                        timeout = setTimeout(function() { func.apply(context, args); }, wait);
                    };
                }
                
                $('#globalSearch').on('input', debounce(function() {
                    table.search($(this).val()).draw();
                }, 400));
            });
        </script>
    </body>
    </html>
    """
    
    # 6. 注入与保存
    html_content = html_template.replace("{{YEARS}}", json.dumps(years_str, ensure_ascii=False))
    html_content = html_content.replace("{{SERIES_DATA}}", json.dumps(series_list, ensure_ascii=False))
    html_content = html_content.replace("{{TABLE_ROWS}}", table_rows)
    
    with open(file_output, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"--- 包含【附件内容】版大屏已更新！ ---")
    print(f"请双击打开：{file_output}")

except Exception as e:
    print(f"运行出错：{e}")

正在读取数据并构建大屏...
--- 包含【附件内容】版大屏已更新！ ---
请双击打开：E:\Library\陈祖武先生受赠图书明细_含附件内容.html


In [10]:
import pandas as pd

# 1. 配置文件路径
file_input = r"E:\Library\赠阅图书整理_时间已提取.xlsx"
# 生成的统计表格的存放路径
file_output = r"E:\Library\陈祖武先生赠书人排行榜.xlsx"

try:
    print("正在扫描数据并计算排行榜...")
    df = pd.read_excel(file_input)
    
    # 清理列名，防止隐藏空格
    df.columns = df.columns.str.replace(r'[\n\r]', '', regex=True).str.strip()
    
    if '送书人' not in df.columns:
        print("致命错误：未找到‘送书人’列！")
    else:
        # 处理册数：如果没有填或者格式不对，默认算作 1 册
        if '册数' in df.columns:
            df['册数'] = pd.to_numeric(df['册数'], errors='coerce').fillna(1)
        else:
            df['册数'] = 1
            
        # 提取数据并清理
        df_subset = df[['送书人', '册数']].dropna(subset=['送书人']).copy()
        df_subset['送书人'] = df_subset['送书人'].astype(str)
        
        # 将逗号统一替换为顿号，并去除多余空格
        df_subset['送书人'] = df_subset['送书人'].str.replace(',', '、').str.replace('，', '、').str.replace(' ', '')
        
        # 拆分多人合捐的情况
        df_subset['送书人'] = df_subset['送书人'].str.split('、')
        # 把合捐名单炸开，每个人独立算一行
        df_exploded = df_subset.explode('送书人')
        
        # 去除拆分后可能产生的空白名字
        df_exploded = df_exploded[df_exploded['送书人'] != '']
        
        # === 核心处理：分组求和与排序 ===
        # 按送书人分组，计算每个人赠送的总册数
        ranking = df_exploded.groupby('送书人')['册数'].sum().reset_index()
        
        # 按册数从大到小排序
        ranking = ranking.sort_values(by='册数', ascending=False).reset_index(drop=True)
        
        # === 整理输出的表格格式 ===
        # 让索引从 1 开始，这就变成了他们的“排名”
        ranking.index = ranking.index + 1
        ranking.reset_index(inplace=True)
        
        # 重命名列名，让表格看起来更专业
        ranking.rename(columns={'index': '排名', '送书人': '学者/机构名称', '册数': '总赠书册数'}, inplace=True)
        
        # 导出为全新的 Excel 表格
        ranking.to_excel(file_output, index=False)
        
        print("="*45)
        print("🎉 统计与导出成功！")
        print(f"本次共统计出 【 {len(ranking)} 】 位独立的赠书人。")
        print(f"完整的排行榜表格已保存至：\n{file_output}")
        print("="*45)

except PermissionError:
    print("错误：文件可能正在被打开。请关闭相关的 Excel 文件后再运行！")
except FileNotFoundError:
    print("错误：找不到文件，请检查文件路径是否正确。")
except Exception as e:
    print(f"运行出错：{e}")

正在扫描数据并计算排行榜...
🎉 统计与导出成功！
本次共统计出 【 395 】 位独立的赠书人。
完整的排行榜表格已保存至：
E:\Library\陈祖武先生赠书人排行榜.xlsx


In [5]:
import pandas as pd
import json

# 1. 配置文件路径
file_input = r"E:\Library\赠阅图书整理_时间已提取.xlsx"
file_output = r"E:\Library\陈祖武先生受赠图书明细_含附件内容.html"

try:
    print("正在读取数据并构建大屏...")
    df = pd.read_excel(file_input)
    
    # 2. 数据清洗
    df = df.dropna(subset=['年', '关系分类'])
    df['年'] = pd.to_numeric(df['年'], errors='coerce')
    df = df.dropna(subset=['年'])
    df = df[(df['年'] >= 1950) & (df['年'] <= 2050)]
    df['年'] = df['年'].astype(int)
    
    # 智能合成标准化日期
    def synthesize_clean_date(row):
        y = f"{int(row['年'])}年"
        m_val = row.get('月', pd.NA)
        m = ""
        if pd.notna(m_val) and str(m_val).strip() != "":
            try:
                m = f"{int(float(str(m_val)))}月"
            except ValueError:
                m = f"{str(m_val).strip()}月"
                
        d_val = row.get('日', pd.NA)
        d = ""
        if pd.notna(d_val) and str(d_val).strip() != "":
            try:
                d = f"{int(float(str(d_val)))}日"
            except ValueError:
                d = f"{str(d_val).strip()}日"
                
        return f"{y}{m}{d}"

    df['标准化时间'] = df.apply(synthesize_clean_date, axis=1)
    
    # 3. 准备图表数据
    relations = ["师长/长辈", "友朋", "晚辈/门生", "官方/机构", "待核对"]
    color_dict = {
        "师长/长辈": "#FF9900", 
        "友朋": "#3366CC",      
        "晚辈/门生": "#109618",   
        "官方/机构": "#990099",
        "待核对": "#999999"
    }

    min_year = df['年'].min()
    max_year = df['年'].max()
    years = list(range(int(min_year), int(max_year) + 1))
    years_str = [str(y) for y in years]
    
    series_list = []
    total_counts = [0] * len(years)
    
    for rel in relations:
        rel_data = []
        for i, year in enumerate(years):
            count = len(df[(df['年'] == year) & (df['关系分类'] == rel)])
            rel_data.append(count)
            total_counts[i] += count
            
        series_list.append({
            "name": rel,
            "type": 'bar',
            "stack": 'Total',
            "itemStyle": {"color": color_dict.get(rel, "#999999")},
            "data": rel_data
        })
        
    series_list.append({
        "name": '年度总册数',
        "type": 'line',
        "smooth": True,
        "symbolSize": 8,
        "itemStyle": {"color": "#333333"},
        "lineStyle": {"width": 3, "type": "dashed"},
        "data": total_counts
    })
    
    # 4. 准备底部列表数据
    table_rows = ""
    df_sorted = df.sort_values(by=['年', '月', '日'], ascending=[True, True, True], na_position='first')
    
    for _, row in df_sorted.iterrows():
        year = str(row['年'])
        relation = str(row['关系分类']).strip() if pd.notna(row['关系分类']) else ""
        sender = str(row['送书人']).strip() if pd.notna(row['送书人']) else ""
        title = str(row['题名']).strip() if pd.notna(row['题名']) else ""
        time_str = str(row['标准化时间'])
        
        # 【新增提取代码】：安全地提取“附件内容”列
        attachment = str(row['附件内容']).strip() if '附件内容' in row and pd.notna(row['附件内容']) else ""
        
        # 【修改 HTML 拼接】：把附件内容加到最后面
        table_rows += f"<tr><td>{year}</td><td>{relation}</td><td>{sender}</td><td>{title}</td><td>{time_str}</td><td>{attachment}</td></tr>\n"

    # 5. HTML 前端代码模板
    html_template = """
    <!DOCTYPE html>
    <html lang="zh-CN">
    <head>
        <meta charset="utf-8">
        <title>陈祖武先生受赠图书明细</title>
        <script src="https://cdnjs.cloudflare.com/ajax/libs/echarts/5.4.2/echarts.min.js"></script>
        <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
        <script src="https://code.jquery.com/jquery-3.6.0.min.js"></script>
        <script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
        
        <style>
            body { font-family: 'Microsoft YaHei', sans-serif; background-color: #f0f2f5; margin: 0; padding: 30px; }
            .container { width: 95%; max-width: 1600px; margin: 0 auto; background: white; padding: 40px; box-shadow: 0 4px 12px rgba(0,0,0,0.05); border-radius: 12px; }
            h1 { text-align: center; color: #2c3e50; margin-bottom: 40px; font-size: 28px; }
            #chart { width: 100%; height: 550px; margin-bottom: 50px; cursor: pointer; }
            .table-container { border-top: 2px dashed #eee; padding-top: 30px; }
            h2 { text-align: center; color: #34495e; margin-bottom: 20px; font-size: 22px; }
            table.dataTable { border-collapse: collapse; width: 100%; }
            table.dataTable thead th { background-color: #f8f9fa; color: #333; }
            
            /* 适度控制一下附件内容列的宽度，防止文字太长撑爆屏幕 */
            table.dataTable td:nth-child(6) { max-width: 400px; white-space: normal; word-wrap: break-word; line-height: 1.5; color: #555; }
            
            .interaction-board {
                display: flex;
                justify-content: space-between;
                align-items: center;
                background: #f8f9fa;
                padding: 15px 25px;
                border-radius: 8px;
                border: 1px solid #e9ecef;
                border-left: 5px solid #3b6291;
                margin-bottom: 20px;
            }
            .status-text { font-size: 16px; color: #2c3e50; }
            .highlight-year { color: #d94e5d; font-weight: bold; font-size: 18px; margin: 0 5px; }
            
            #resetBtn {
                background-color: #3b6291; color: white; border: none; padding: 8px 20px;
                border-radius: 4px; cursor: pointer; font-size: 14px; font-weight: bold;
                transition: all 0.3s; display: none; 
            }
            #resetBtn:hover { background-color: #2c4a6e; box-shadow: 0 2px 5px rgba(0,0,0,0.2); }
            
            .dataTables_filter { display: none; }
            .global-search { 
                padding: 8px 12px; border: 1px solid #ced4da; border-radius: 4px; 
                outline: none; width: 220px; transition: border-color 0.3s;
            }
            .global-search:focus { border-color: #3b6291; }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>陈祖武先生受赠图书明细</h1>
            <div id="chart" title="点击柱子查看当年明细"></div>
            
            <div class="table-container">
                <h2>赠阅明细检索</h2>
                
                <div class="interaction-board">
                    <div class="status-text">
                        <span>💡 <strong>操作提示：</strong>点击上方图表中的年份柱子即可过滤。</span>
                        <span style="margin-left:20px;">当前查看：<span id="currentYearDisplay" class="highlight-year">所有年份</span></span>
                    </div>
                    <div>
                        <input type="text" id="globalSearch" class="global-search" placeholder="🔍 检索书名/人名/题词...">
                        <button id="resetBtn">🔄 显示全部年份</button>
                    </div>
                </div>
                
                <table id="myTable" class="display">
                    <thead>
                        <tr>
                            <th>年份</th>
                            <th>关系分类</th>
                            <th>送书人</th>
                            <th>赠书书名</th>
                            <th>规范赠阅时间</th>
                            <th>附件内容</th>
                        </tr>
                    </thead>
                    <tbody>
                        {{TABLE_ROWS}}
                    </tbody>
                </table>
            </div>
        </div>

        <script>
            var chartDom = document.getElementById('chart');
            var myChart = echarts.init(chartDom);
            
            var option = {
                tooltip: { trigger: 'axis', axisPointer: { type: 'cross' } },
                legend: { top: 'bottom' },
                grid: { left: '3%', right: '4%', bottom: '10%', containLabel: true },
                dataZoom: [ { type: 'inside' }, { type: 'slider', bottom: '0%' } ],
                xAxis: { type: 'category', data: {{YEARS}}, axisLabel: { rotate: 45 } },
                yAxis: { type: 'value', name: '赠书册数 (本)' },
                series: {{SERIES_DATA}}
            };
            myChart.setOption(option);

            $(document).ready(function() {
                if ($.fn.dataTable.ext && $.fn.dataTable.ext.search) {
                    $.fn.dataTable.ext.search = [];
                }
                
                var table = $('#myTable').DataTable({
                    "order": [[ 0, "asc" ]], 
                    "pageLength": 10,
                    "language": { 
                        "processing": "处理中...",
                        "lengthMenu": "显示 _MENU_ 项结果",
                        "zeroRecords": "📭 该年份暂无赠阅记录哦~",
                        "info": "显示第 _START_ 至 _END_ 项结果，共 _TOTAL_ 项",
                        "infoEmpty": "显示第 0 至 0 项结果，共 0 项",
                        "infoFiltered": "(由 _MAX_ 项结果过滤)",
                        "search": "搜索:",
                        "emptyTable": "表中数据为空",
                        "paginate": { "first": "首页", "previous": "上页", "next": "下页", "last": "末页" }
                    }
                });
                
                myChart.on('click', function (params) {
                    var clickYear = params.name; 
                    if (clickYear) {
                        table.column(0).search('^' + clickYear + '$', true, false).draw();
                        
                        $('#currentYearDisplay').text(clickYear + '年');
                        $('#resetBtn').fadeIn(); 
                        
                        $('html, body').animate({
                            scrollTop: $(".table-container").offset().top - 20
                        }, 500);
                    }
                });
                
                $('#resetBtn').on('click', function() {
                    table.column(0).search('').draw(); 
                    $('#currentYearDisplay').text('所有年份');
                    $(this).fadeOut(); 
                });
                
                function debounce(func, wait) {
                    var timeout;
                    return function() {
                        var context = this, args = arguments;
                        clearTimeout(timeout);
                        timeout = setTimeout(function() { func.apply(context, args); }, wait);
                    };
                }
                
                $('#globalSearch').on('input', debounce(function() {
                    table.search($(this).val()).draw();
                }, 400));
            });
        </script>
    </body>
    </html>
    """
    
    # 6. 注入与保存
    html_content = html_template.replace("{{YEARS}}", json.dumps(years_str, ensure_ascii=False))
    html_content = html_content.replace("{{SERIES_DATA}}", json.dumps(series_list, ensure_ascii=False))
    html_content = html_content.replace("{{TABLE_ROWS}}", table_rows)
    
    with open(file_output, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"--- 包含【附件内容】版大屏已更新！ ---")
    print(f"请双击打开：{file_output}")

except Exception as e:
    print(f"运行出错：{e}")

正在读取数据并构建大屏...
--- 包含【附件内容】版大屏已更新！ ---
请双击打开：E:\Library\陈祖武先生受赠图书明细_含附件内容.html


In [6]:
import pandas as pd
import json

# 1. 配置文件路径
file_input = r"E:\Library\赠阅图书整理_时间已提取.xlsx"
file_output = r"E:\Library\陈祖武先生受赠图书明细_含附件内容.html"

try:
    print("正在读取数据并构建大屏...")
    df = pd.read_excel(file_input)
    
    # 2. 数据清洗 (极简修改：只剔除没分类的，把没年份的保留给表格)
    df = df.dropna(subset=['关系分类'])
    # 把年份强转为数字，转不了的和原本就是空的变成 NaN
    df['数字年份'] = pd.to_numeric(df['年'], errors='coerce')
    
    # 智能合成标准化日期 (微调一点点容错，遇到没年份的直接输出未知)
    def synthesize_clean_date(row):
        if pd.isna(row['数字年份']):
            return "未知"
            
        y = f"{int(row['数字年份'])}年"
        m_val = row.get('月', pd.NA)
        m = ""
        if pd.notna(m_val) and str(m_val).strip() != "":
            try:
                m = f"{int(float(str(m_val)))}月"
            except ValueError:
                m = f"{str(m_val).strip()}月"
                
        d_val = row.get('日', pd.NA)
        d = ""
        if pd.notna(d_val) and str(d_val).strip() != "":
            try:
                d = f"{int(float(str(d_val)))}日"
            except ValueError:
                d = f"{str(d_val).strip()}日"
                
        return f"{y}{m}{d}"

    df['标准化时间'] = df.apply(synthesize_clean_date, axis=1)
    
    # ==========================================
    # 3. 准备图表数据 (提取出一个专属 df_chart，完全照搬你原来的严格逻辑)
    # ==========================================
    df_chart = df.dropna(subset=['数字年份']).copy()
    df_chart = df_chart[(df_chart['数字年份'] >= 1950) & (df_chart['数字年份'] <= 2050)]
    df_chart['数字年份'] = df_chart['数字年份'].astype(int)

    relations = ["师长/长辈", "友朋", "晚辈/门生", "官方/机构", "待核对"]
    color_dict = {
        "师长/长辈": "#FF9900", 
        "友朋": "#3366CC",      
        "晚辈/门生": "#109618",   
        "官方/机构": "#990099",
        "待核对": "#999999"
    }

    min_year = df_chart['数字年份'].min()
    max_year = df_chart['数字年份'].max()
    years = list(range(int(min_year), int(max_year) + 1))
    years_str = [str(y) for y in years]
    
    series_list = []
    total_counts = [0] * len(years)
    
    for rel in relations:
        rel_data = []
        for i, year in enumerate(years):
            count = len(df_chart[(df_chart['数字年份'] == year) & (df_chart['关系分类'] == rel)])
            rel_data.append(count)
            total_counts[i] += count
            
        series_list.append({
            "name": rel,
            "type": 'bar',
            "stack": 'Total',
            "itemStyle": {"color": color_dict.get(rel, "#999999")},
            "data": rel_data
        })
        
    series_list.append({
        "name": '年度总册数',
        "type": 'line',
        "smooth": True,
        "symbolSize": 8,
        "itemStyle": {"color": "#333333"},
        "lineStyle": {"width": 3, "type": "dashed"},
        "data": total_counts
    })
    
    # ==========================================
    # 4. 准备底部列表数据 (使用完整的 df，把未知年份的全包进去)
    # ==========================================
    table_rows = ""
    # 按照数字年份排序，用 na_position='last' 把没有年份的整齐排到列表最下面
    df_sorted = df.sort_values(by=['数字年份', '月', '日'], ascending=[True, True, True], na_position='last')
    
    for _, row in df_sorted.iterrows():
        # 如果是 NaN 就显示“未知”，否则转成正常的整数年份
        year = "未知" if pd.isna(row['数字年份']) else str(int(row['数字年份']))
        relation = str(row['关系分类']).strip() if pd.notna(row['关系分类']) else ""
        sender = str(row['送书人']).strip() if pd.notna(row['送书人']) else ""
        title = str(row['题名']).strip() if pd.notna(row['题名']) else ""
        time_str = str(row['标准化时间'])
        
        attachment = str(row['附件内容']).strip() if '附件内容' in row and pd.notna(row['附件内容']) else ""
        
        table_rows += f"<tr><td>{year}</td><td>{relation}</td><td>{sender}</td><td>{title}</td><td>{time_str}</td><td>{attachment}</td></tr>\n"

    # 5. HTML 前端代码模板 (未做任何更改)
    html_template = """
    <!DOCTYPE html>
    <html lang="zh-CN">
    <head>
        <meta charset="utf-8">
        <title>陈祖武先生受赠图书明细</title>
        <script src="https://cdnjs.cloudflare.com/ajax/libs/echarts/5.4.2/echarts.min.js"></script>
        <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
        <script src="https://code.jquery.com/jquery-3.6.0.min.js"></script>
        <script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
        
        <style>
            body { font-family: 'Microsoft YaHei', sans-serif; background-color: #f0f2f5; margin: 0; padding: 30px; }
            .container { width: 95%; max-width: 1600px; margin: 0 auto; background: white; padding: 40px; box-shadow: 0 4px 12px rgba(0,0,0,0.05); border-radius: 12px; }
            h1 { text-align: center; color: #2c3e50; margin-bottom: 40px; font-size: 28px; }
            #chart { width: 100%; height: 550px; margin-bottom: 50px; cursor: pointer; }
            .table-container { border-top: 2px dashed #eee; padding-top: 30px; }
            h2 { text-align: center; color: #34495e; margin-bottom: 20px; font-size: 22px; }
            table.dataTable { border-collapse: collapse; width: 100%; }
            table.dataTable thead th { background-color: #f8f9fa; color: #333; }
            
            table.dataTable td:nth-child(6) { max-width: 400px; white-space: normal; word-wrap: break-word; line-height: 1.5; color: #555; }
            
            .interaction-board {
                display: flex;
                justify-content: space-between;
                align-items: center;
                background: #f8f9fa;
                padding: 15px 25px;
                border-radius: 8px;
                border: 1px solid #e9ecef;
                border-left: 5px solid #3b6291;
                margin-bottom: 20px;
            }
            .status-text { font-size: 16px; color: #2c3e50; }
            .highlight-year { color: #d94e5d; font-weight: bold; font-size: 18px; margin: 0 5px; }
            
            #resetBtn {
                background-color: #3b6291; color: white; border: none; padding: 8px 20px;
                border-radius: 4px; cursor: pointer; font-size: 14px; font-weight: bold;
                transition: all 0.3s; display: none; 
            }
            #resetBtn:hover { background-color: #2c4a6e; box-shadow: 0 2px 5px rgba(0,0,0,0.2); }
            
            .dataTables_filter { display: none; }
            .global-search { 
                padding: 8px 12px; border: 1px solid #ced4da; border-radius: 4px; 
                outline: none; width: 220px; transition: border-color 0.3s;
            }
            .global-search:focus { border-color: #3b6291; }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>陈祖武先生受赠图书明细</h1>
            <div id="chart" title="点击柱子查看当年明细"></div>
            
            <div class="table-container">
                <h2>赠阅明细检索</h2>
                
                <div class="interaction-board">
                    <div class="status-text">
                        <span>💡 <strong>操作提示：</strong>点击上方图表中的年份柱子即可过滤。</span>
                        <span style="margin-left:20px;">当前查看：<span id="currentYearDisplay" class="highlight-year">所有年份</span></span>
                    </div>
                    <div>
                        <input type="text" id="globalSearch" class="global-search" placeholder="🔍 检索书名/人名/题词...">
                        <button id="resetBtn">🔄 显示全部年份</button>
                    </div>
                </div>
                
                <table id="myTable" class="display">
                    <thead>
                        <tr>
                            <th>年份</th>
                            <th>关系分类</th>
                            <th>送书人</th>
                            <th>赠书书名</th>
                            <th>规范赠阅时间</th>
                            <th>附件内容</th>
                        </tr>
                    </thead>
                    <tbody>
                        {{TABLE_ROWS}}
                    </tbody>
                </table>
            </div>
        </div>

        <script>
            var chartDom = document.getElementById('chart');
            var myChart = echarts.init(chartDom);
            
            var option = {
                tooltip: { trigger: 'axis', axisPointer: { type: 'cross' } },
                legend: { top: 'bottom' },
                grid: { left: '3%', right: '4%', bottom: '10%', containLabel: true },
                dataZoom: [ { type: 'inside' }, { type: 'slider', bottom: '0%' } ],
                xAxis: { type: 'category', data: {{YEARS}}, axisLabel: { rotate: 45 } },
                yAxis: { type: 'value', name: '赠书册数 (本)' },
                series: {{SERIES_DATA}}
            };
            myChart.setOption(option);

            $(document).ready(function() {
                if ($.fn.dataTable.ext && $.fn.dataTable.ext.search) {
                    $.fn.dataTable.ext.search = [];
                }
                
                var table = $('#myTable').DataTable({
                    "order": [[ 0, "asc" ]], 
                    "pageLength": 10,
                    "language": { 
                        "processing": "处理中...",
                        "lengthMenu": "显示 _MENU_ 项结果",
                        "zeroRecords": "📭 该年份暂无赠阅记录哦~",
                        "info": "显示第 _START_ 至 _END_ 项结果，共 _TOTAL_ 项",
                        "infoEmpty": "显示第 0 至 0 项结果，共 0 项",
                        "infoFiltered": "(由 _MAX_ 项结果过滤)",
                        "search": "搜索:",
                        "emptyTable": "表中数据为空",
                        "paginate": { "first": "首页", "previous": "上页", "next": "下页", "last": "末页" }
                    }
                });
                
                myChart.on('click', function (params) {
                    var clickYear = params.name; 
                    if (clickYear) {
                        table.column(0).search('^' + clickYear + '$', true, false).draw();
                        
                        $('#currentYearDisplay').text(clickYear + '年');
                        $('#resetBtn').fadeIn(); 
                        
                        $('html, body').animate({
                            scrollTop: $(".table-container").offset().top - 20
                        }, 500);
                    }
                });
                
                $('#resetBtn').on('click', function() {
                    table.column(0).search('').draw(); 
                    $('#currentYearDisplay').text('所有年份');
                    $(this).fadeOut(); 
                });
                
                function debounce(func, wait) {
                    var timeout;
                    return function() {
                        var context = this, args = arguments;
                        clearTimeout(timeout);
                        timeout = setTimeout(function() { func.apply(context, args); }, wait);
                    };
                }
                
                $('#globalSearch').on('input', debounce(function() {
                    table.search($(this).val()).draw();
                }, 400));
            });
        </script>
    </body>
    </html>
    """
    
    # 6. 注入与保存
    html_content = html_template.replace("{{YEARS}}", json.dumps(years_str, ensure_ascii=False))
    html_content = html_content.replace("{{SERIES_DATA}}", json.dumps(series_list, ensure_ascii=False))
    html_content = html_content.replace("{{TABLE_ROWS}}", table_rows)
    
    with open(file_output, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"--- 包含【附件内容】版大屏已更新！ ---")
    print(f"请双击打开：{file_output}")

except Exception as e:
    print(f"运行出错：{e}")

正在读取数据并构建大屏...
--- 包含【附件内容】版大屏已更新！ ---
请双击打开：E:\Library\陈祖武先生受赠图书明细_含附件内容.html
